**[Source]** Donghwan Project (오류 유형 분류: 부분교정/잘못된 교정/미교정/과교정/내용 추가·삭제) + Jisoo Project 11단계 진단(왕복 변환·공백·반복·길이)
**[Status]** ADAPTED
**[Role]** 선택 모델의 Validation 오류를 유형별로 분류하고 빈도·조건·대표 사례·원인 후보를 정리
**[Modification]** 동환의 분류 기준을 수식(편집 거리)으로 명확히 정의해 자동 분류. 지수의 왕복 변환(NFKC로 동일) 진단을 별도 유형으로 분리. 사례는 Validation에서만 뽑는다.
**Test 사례는 보지 않는다. 오류 분석 결과로 최종 설정을 바꿀 때는 Validation에서 재검증한 뒤 13번에 기록한다.**

# 12. 오류 분석 (Validation, pko-T5 Greedy)
**분류 기준(상호 배타, 우선순위 순서)** — S=입력, T=정답, P=예측(모두 NFKC 정규화 후 비교; Lev=편집 거리):
1. **정상**: P == T
2. **형식 변환만(raw 오답)**: NFKC 전에는 P≠T이지만 NFKC 후 같음 → tokenizer 왕복 변환 등 문자 표현 차이. 의미상 오류는 아니지만 실제 출력 문자는 바뀌므로 별도 집계
3. **과교정**: S==T인데 P≠S (고칠 것이 없는 문장을 바꿈)
4. **미교정**: S≠T이고 P==S (아무것도 고치지 않음)
5. **내용 추가·삭제**: 그 외이면서 |len(P)−len(T)|/len(T) > 0.3
6. **부분교정**: 그 외이면서 Lev(P,T) < Lev(S,T) (정답에 가까워졌으나 불완전)
7. **잘못된 교정**: 그 외(Lev(P,T) ≥ Lev(S,T): 정답에서 멀어지거나 그대로 멀다)
※ 이 자동 분류는 규칙 기반 근사이며 사람이 하나씩 검수한 것이 아니다. 특히 ‘부분교정’과 ‘잘못된 교정’의 경계는 편집 거리에 의존한다.

In [1]:
# [공통 준비] 경로 · 재현성 · 05번 검증 통과 확인
import os, sys, json, re, time, math, random, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 70); pd.set_option("display.unicode.east_asian_width", True)

def _find_root():
    p = Path.cwd().resolve()
    for c in [p, *p.parents]:
        if (c / "config" / "paths.json").exists():
            return c
    raise FileNotFoundError("config/paths.json이 있는 통합 프로젝트 루트를 찾지 못했습니다(notebooks 폴더에서 실행하세요).")
ROOT = _find_root(); sys.path.insert(0, str(ROOT / "src"))
import common, ko_metrics as km
P = common.load_paths(ROOT)
SEED = 42; random.seed(SEED); np.random.seed(SEED)

VER = json.loads((P.PROCESSED / "verification_05" / "verification_result.json").read_text(encoding="utf-8"))
assert VER["verdict"] == "PASS", "05번 전처리 검증이 PASS가 아닙니다 → 모델링을 진행하지 않습니다"
MAN = common.manifest(P)
assert VER["sha256_actual"]["train"] == MAN["sha256"]["train.jsonl"], "검증 이후 데이터가 바뀌었습니다(05번을 다시 실행하세요)"
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| numpy", np.__version__, "|", platform.platform())
print("통합 프로젝트:", ROOT); print("지수 최종 데이터:", P.DATA_DIR, "|", MAN["dataset_version"])
print("05번 검증:", VER["verdict"], "@", VER["verified_at"], "| metric backends:", km.BACKENDS)

Python 3.10.12 | pandas 2.3.3 | numpy 2.2.6 | Linux-6.8.0-138-generic-x86_64-with-glibc2.35
통합 프로젝트: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/integrated_korean_correction
지수 최종 데이터: /sessions/rcw-01co1f5gwjl7cyyghugs6woe/mnt/pro4_team3/data/preprocessed_final | preprocessed_final_v1
05번 검증: PASS @ 2026-09-21 05:30:57 | metric backends: {'rapidfuzz': False, 'sacrebleu': False}


In [2]:
# [셀 1] 로드 + 분류
VAL = common.read_split(P, "validation", columns=["document_id", "utterance_id", "input", "target", "change_type", "flag_seen_input_in_train", "flag_digit_changed", "flag_alpha_changed", "flag_emoji_changed", "flag_form_conflict", "flag_missed_correction_suspect", "flag_partial_correction_suspect"])
SRC, TGT, DOC = [r["input"] for r in VAL], [r["target"] for r in VAL], [r["document_id"] for r in VAL]
rows = common.read_jsonl(P.J_OUT / "decoding_comparison" / "val_predictions_greedy.jsonl")
assert [r["utterance_id"] for r in rows] == [r["utterance_id"] for r in VAL] and [r["target"] for r in rows] == TGT
PRED = [str(r["prediction"]) for r in rows]; N = km.normalize_for_evaluation
def classify(s, t, p):
    ns, nt, np_ = N(s), N(t), N(p)
    if np_ == nt: return "정상" if p == t else "형식 변환만(raw 오답)"
    if ns == nt: return "과교정"
    if np_ == ns: return "미교정"
    if abs(len(np_) - len(nt)) / max(1, len(nt)) > 0.3: return "내용 추가·삭제"
    return "부분교정" if km.levenshtein(np_, nt) < km.levenshtein(ns, nt) else "잘못된 교정"
t0 = time.time(); CAT = np.array([classify(s, t, p) for s, t, p in zip(SRC, TGT, PRED)]); print("분류 완료 %.0f초" % (time.time() - t0))
order = ["정상", "형식 변환만(raw 오답)", "과교정", "미교정", "부분교정", "잘못된 교정", "내용 추가·삭제"]
cnt = pd.Series(CAT).value_counts().reindex(order).fillna(0).astype(int); TB = pd.DataFrame({"행 수": cnt, "전체 대비 %": (100 * cnt / len(CAT)).round(2), "오류(정상 제외) 대비 %": (100 * cnt / (len(CAT) - cnt["정상"])).round(2)}); TB.loc["정상", "오류(정상 제외) 대비 %"] = np.nan
print(TB.to_string()); print("\n※ ‘형식 변환만’을 오류로 볼지는 용도에 따라 다르다. 아래에서는 ‘실질 오류’ = 형식 변환만·정상 제외로 따로 센다.")
REAL = ~np.isin(CAT, ["정상", "형식 변환만(raw 오답)"]); print("실질 오류 행: %d (%.2f%%)" % (REAL.sum(), 100 * REAL.mean()))

분류 완료 2초
                       행 수  전체 대비 %  오류(정상 제외) 대비 %
정상                   28838        52.69                     NaN
형식 변환만(raw 오답)   9281        16.96                   35.85
과교정                   956         1.75                    3.69
미교정                  1369         2.50                    5.29
부분교정               10651        19.46                   41.14
잘못된 교정             3286         6.00                   12.69
내용 추가·삭제           349         0.64                    1.35

※ ‘형식 변환만’을 오류로 볼지는 용도에 따라 다르다. 아래에서는 ‘실질 오류’ = 형식 변환만·정상 제외로 따로 센다.
실질 오류 행: 16611 (30.35%)


In [3]:
# [셀 2] 오류 유형이 반복되는 입력 조건 — 길이, 정답 편집량, change_type
need = np.array([s != t for s, t in zip(SRC, TGT)]); lens = np.array([len(s) for s in SRC]); edit = np.array([km.levenshtein(N(s), N(t)) for s, t in zip(SRC, TGT)])
def rate_by(keys, name):
    d = pd.DataFrame({name: keys, "실질오류": REAL}); g = d.groupby(name, observed=True)["실질오류"].agg(["size", "mean"]); g["mean"] = (100 * g["mean"]).round(2); g.columns = ["행 수", "실질 오류율 %"]; return g
print(rate_by(pd.cut(lens, [-1, 5, 10, 20, 40, 10**6], labels=["≤5", "6-10", "11-20", "21-40", ">40"]), "입력 길이(문자)").to_string()); print()
print(rate_by(pd.cut(edit, [-1, 0, 1, 2, 4, 10**6], labels=["0(교정 없음)", "1", "2", "3-4", "≥5"]), "정답 편집 거리").to_string()); print()
ct = pd.Series([r["change_type"] for r in VAL]); print(rate_by(ct.values, "change_type").sort_values("행 수", ascending=False).head(10).to_string())
print("\n유형별 평균 입력 길이:"); print(pd.Series(lens).groupby(CAT).mean().round(1).reindex(order).to_string())

                 행 수  실질 오류율 %
입력 길이(문자)                      
≤5               11812          26.78
6-10             14077          26.04
11-20            18407          29.89
21-40             8979          38.63
>40               1455          55.81

                행 수  실질 오류율 %
정답 편집 거리                      
0(교정 없음)     7396          12.93
1               14426          23.42
2               12194          27.48
3-4             13613          37.47
≥5               7101          53.85

                  행 수  실질 오류율 %
change_type                           
punct_only        28378          27.46
spelling_or_word  14135          47.58
unchanged          7367          12.91
spacing_only       4850          23.55

유형별 평균 입력 길이:
정상                     11.5
형식 변환만(raw 오답)    15.9
과교정                   12.1
미교정                    9.6
부분교정                 18.8
잘못된 교정              11.1
내용 추가·삭제           15.1


In [4]:
# [셀 3] 반복·길이·빈 출력 진단 (지수 진단과 같은 목적, 정의는 여기서 명시)
import re
rep = re.compile(r"(.{2,}?)\1{3,}")
diag = {"빈 출력": sum(1 for p in PRED if not p.strip()), "정답보다 150% 초과로 긴 출력": sum(1 for p, t in zip(PRED, TGT) if len(p) > 1.5 * max(1, len(t))), "정답보다 50% 미만으로 짧은 출력": sum(1 for p, t in zip(PRED, TGT) if len(p) < 0.5 * len(t)),
        "반복 패턴(정답엔 없음)": sum(1 for p, t in zip(PRED, TGT) if rep.search(p) and not rep.search(t))}
print(pd.Series(diag).to_string()); print("※ 지수 기록(전체 Validation, Greedy): 빈 출력 0, 150% 초과 128, 생성 상한 도달 후보 59, 반복 후보 10 — 정의가 조금 다를 수 있음")

빈 출력                              0
정답보다 150% 초과로 긴 출력       128
정답보다 50% 미만으로 짧은 출력     38
반복 패턴(정답엔 없음)               9
※ 지수 기록(전체 Validation, Greedy): 빈 출력 0, 150% 초과 128, 생성 상한 도달 후보 59, 반복 후보 10 — 정의가 조금 다를 수 있음


In [5]:
# [셀 4] 데이터 라벨 문제 후보와 모델 오류의 관계 — 오류가 모델 탓인지 정답 라벨 탓인지 구분
LAB = {"flag_digit_changed": "숫자가 바뀐 라벨", "flag_alpha_changed": "영문이 바뀐 라벨", "flag_emoji_changed": "이모지가 바뀐 라벨", "flag_form_conflict": "같은 입력에 다른 정답(충돌)", "flag_missed_correction_suspect": "교정 누락 의심 라벨", "flag_partial_correction_suspect": "부분 교정 의심 라벨"}
out = []
for f, nm in LAB.items():
    m = np.array([bool(r[f]) for r in VAL]); out.append({"플래그": nm, "행 수": int(m.sum()), "실질 오류율 %": round(100 * REAL[m].mean(), 2) if m.any() else np.nan})
out.append({"플래그": "(전체)", "행 수": len(VAL), "실질 오류율 %": round(100 * REAL.mean(), 2)}); print(pd.DataFrame(out).to_string(index=False))
print("\n※ 플래그가 있는 행의 오류율이 전체보다 높다면 그 오류의 일부는 모델이 아니라 정답 라벨의 문제일 수 있다. 라벨을 실제로 고치는 것은 별도 라벨 감사 단계이며, 이 노트북은 데이터를 수정하지 않는다.")

                     플래그  행 수  실질 오류율 %
           숫자가 바뀐 라벨     74          79.73
           영문이 바뀐 라벨     12          91.67
         이모지가 바뀐 라벨     41          68.29
같은 입력에 다른 정답(충돌)   5550          24.90
        교정 누락 의심 라벨      1         100.00
        부분 교정 의심 라벨      2         100.00
                     (전체)  54730          30.35

※ 플래그가 있는 행의 오류율이 전체보다 높다면 그 오류의 일부는 모델이 아니라 정답 라벨의 문제일 수 있다. 라벨을 실제로 고치는 것은 별도 라벨 감사 단계이며, 이 노트북은 데이터를 수정하지 않는다.


In [6]:
# [셀 5] 대표 실패 사례 (Validation, seed=42로 유형별 무작위 추출 — 잘 된 것만 고르지 않음)
rng = np.random.default_rng(42); SHOW = {}
for c in ["과교정", "미교정", "부분교정", "잘못된 교정", "내용 추가·삭제", "형식 변환만(raw 오답)"]:
    idx = np.where(CAT == c)[0]
    if len(idx) == 0: continue
    pick = rng.choice(idx, size=min(4, len(idx)), replace=False); SHOW[c] = pick
    print(f"\n■ {c} ({len(idx):,}건 중 무작위 {len(pick)}건)")
    for i in pick: print(f"  입력: {SRC[i]}\n  정답: {TGT[i]}\n  예측: {PRED[i]}\n  [{VAL[i]['utterance_id']}]\n")


■ 과교정 (956건 중 무작위 4건)
  입력: name1
  정답: name1
  예측: name1,
  [MDRW2100001387.26.1]

  입력: 아기 = 천사
  정답: 아기 = 천사
  예측: 아기 = 천사.
  [MDRW2100000119.8.30]

  입력: 급기야 좀 퀴어프렌들리한 분위기도 있고(...) 아무래도 소수자들에 대해서 관심이 많은 편이지 과 자체가....
  정답: 급기야 좀 퀴어프렌들리한 분위기도 있고(...) 아무래도 소수자들에 대해서 관심이 많은 편이지 과 자체가....
  예측: 급기야 좀 퀴어 프렌들리한 분위기도 있고(...) 아무래도 소수자들에 대해서 관심이 많은 편이지 과 자체가...
  [MDRW2100025157.1.24]

  입력: 불고기 먹었어요 ㅋㅋㅋㅋ
  정답: 불고기 먹었어요 ㅋㅋㅋㅋ
  예측: 불고기 먹었어요. ᄏᄏᄏᄏ
  [MDRW2100034086.1.22]


■ 미교정 (1,369건 중 무작위 4건)
  입력: 걍 밑반찬이랑 적당히 그렇게
  정답: 그냥 밑반찬이랑 적당히 그렇게
  예측: 걍 밑반찬이랑 적당히 그렇게
  [MDRW2100024688.1.6]

  입력: 야심차게 준비했는데
  정답: 야심 차게 준비했는데.
  예측: 야심차게 준비했는데
  [MDRW2100000255.26.17]

  입력: 죽을래!
  정답: 죽을래?
  예측: 죽을래!
  [MDRW2100000130.28.4]

  입력: 거짓말!
  정답: 거짓말
  예측: 거짓말!
  [MDRW2100006547.1.22]


■ 부분교정 (10,651건 중 무작위 4건)
  입력: 그죸ㅋㅋㅋㅋㅋ시그마...함수.. 수학 재밌긴했는데..
  정답: 그죠. ㅋㅋㅋㅋㅋㅋ 시그마... 함수... 수학 재밌긴 했는데...
  예측: 그쵸. ᄏᄏᄏᄏᄏᄏ 시그마... 함수... 수학 재밌긴 했는데...
  [MDRW2100029961.1.15]

  입력: 맞아요 싸이코 같은 앨범 또 나오면 좋겠어요
  정답: 맞아요. 사

### 후처리 실험 — 홀로 쓰인 자모(ㅋㅋ, ㅠㅠ 등) 복원
오류 분석에서 ‘형식 변환만’이 매우 많고 사례(예: 정답 `ㅋㅋㅋ` ↔ 예측 `ᄏᄏᄏ`)에서 원인이 보인다. pko-T5 tokenizer(SentencePiece)가 입력을 **NFKC 정규화**하기 때문에 호환 자모 `ㅋ`(U+314B)가 초성 `ᄏ`(U+110F)로 바뀌어 출력에도 나온다. 화면에서는 비슷해 보여도 **다른 문자열**이다.
가설: 완성형 음절로 합쳐지지 않고 남은 옛 한글 자모를 호환 자모로 되돌리면 raw 기준 오답이 줄어든다(모델·데이터는 바꾸지 않는 결정적 규칙, `src/ko_postprocess.py`). 이 가설은 **Validation에서만** 검증하며, 채택 여부는 13번에서 결정한다. 한계: `ㅇㅡ`처럼 자모 여러 개가 NFKC에서 음절로 합쳐진 경우는 되돌릴 수 없고, `ㅀ`·`ㅄ`는 규칙이 처리하지 못한다.

In [7]:
# [셀 5b] 후처리 실험 (Validation)
from ko_postprocess import restore_compat_jamo
PP = [restore_compat_jamo(p) for p in PRED]; chg = sum(1 for a, b in zip(PRED, PP) if a != b); print("후처리로 문자열이 바뀐 예측:", chg, "행 (%.2f%%)" % (100 * chg / len(PRED)))
mr, mp = km.generation_metrics(SRC, TGT, PRED, nfkc=False, with_chrf=False), km.generation_metrics(SRC, TGT, PP, nfkc=False, with_chrf=False)
mn = km.generation_metrics(SRC, TGT, PRED, nfkc=True, with_chrf=False)
print(pd.DataFrame({"raw, 후처리 전": mr, "raw, 후처리 후": mp, "NFKC(참고, 후처리 무관)": mn}).T[["exact_match", "need_correction_em", "unchanged_em", "balanced_em", "cer", "over_correction_rate", "miss_rate"]].round(4).to_string())
exr = np.array([p == t for p, t in zip(PRED, TGT)]); exp = np.array([p == t for p, t in zip(PP, TGT)])
d = km.boot_balanced_em_diff(exp, exr, need, DOC); print("raw Balanced EM 개선(후처리 후−전): %.4f (CI95 %.4f ~ %.4f)" % d)
worse = int((exr & ~exp).sum()); better = int((~exr & exp).sum()); print("정답이었다가 틀리게 된 행:", worse, "| 틀렸다가 정답이 된 행:", better)
# 후처리 후에도 raw로 다른 행 중 NFKC로는 같은 행 = 아직 남은 형식 변환
rest = int(((np.array([p != t for p, t in zip(PP, TGT)])) & np.array([N(p) == N(t) for p, t in zip(PP, TGT)])).sum()); print("후처리 후에도 남은 ‘형식 변환만’ 행:", rest)
POST = {"function": "ko_postprocess.restore_compat_jamo", "rows_changed": chg, "raw_metrics_before": mr, "raw_metrics_after": mp, "balanced_em_gain_ci95": list(map(float, d)), "worse_rows": worse, "better_rows": better, "remaining_format_only": rest, "scope": "Validation only"}

후처리로 문자열이 바뀐 예측: 12893 행 (23.56%)
                         exact_match  need_correction_em  unchanged_em  balanced_em     cer  over_correction_rate  miss_rate
raw, 후처리 전                0.5269              0.5129        0.6169       0.5649  0.1025                0.3831     0.0275
raw, 후처리 후                0.6896              0.6616        0.8701       0.7658  0.0374                0.1299     0.0293
NFKC(참고, 후처리 무관)       0.6965              0.6693        0.8707       0.7700  0.0355                0.1293     0.0289
raw Balanced EM 개선(후처리 후−전): 0.2009 (CI95 0.1925 ~ 0.2091)
정답이었다가 틀리게 된 행: 3 | 틀렸다가 정답이 된 행: 8908
후처리 후에도 남은 ‘형식 변환만’ 행: 376


In [8]:
# [셀 5c] 실질 오류의 세부 성격 — 띄어쓰기/문장부호 차이만 있는 오류는 얼마나 되는가 (후처리 적용 예측 기준)
import re
strip_sp = lambda x: re.sub(r"\s+", "", x); strip_pu = lambda x: re.sub(r"[^\w\sㄱ-ㆎ]", "", x)   # 문장부호 제거(한글·영숫자·호환자모·공백은 유지)
def kind(p, t):
    p, t = N(p), N(t)
    if p == t: return "정상"
    if strip_sp(p) == strip_sp(t): return "띄어쓰기만 다름"
    if strip_pu(p) == strip_pu(t): return "문장부호만 다름"
    if strip_sp(strip_pu(p)) == strip_sp(strip_pu(t)): return "띄어쓰기+문장부호만 다름"
    return "글자(어휘) 차이 있음"
K = np.array([kind(p, t) for p, t in zip(PP, TGT)]); kc = pd.Series(K).value_counts(); tot_err = int((K != "정상").sum())
print("후처리 적용 후 오답 행:", tot_err, "(%.2f%%)" % (100 * tot_err / len(K)))
print(pd.DataFrame({"행 수": kc, "오답 대비 %": (100 * kc / tot_err).round(2)}).drop(index="정상").to_string())
# 문장부호 차이가 특히 큰 위치: 문장 끝 부호
def last_punct(x): return x.strip()[-1] if x.strip() and not re.match(r"[\wㄱ-ㆎ]", x.strip()[-1]) else "(없음)"
pe = [i for i in range(len(K)) if K[i] == "문장부호만 다름"]
from collections import Counter
print("\n문장부호만 다른 오답의 (정답 끝부호 → 예측 끝부호) 상위 10:", Counter((last_punct(TGT[i]), last_punct(PP[i])) for i in pe).most_common(10))
same_in = {}
for i in range(len(K)): same_in.setdefault(SRC[i], set()).add(TGT[i])
conf = np.array([len(same_in[s]) > 1 for s in SRC]); print("\n같은 입력에 서로 다른 정답이 Validation 안에 존재하는 행: %d (%.2f%%) | 그 행의 오답률 %.2f%% vs 나머지 %.2f%%" % (conf.sum(), 100 * conf.mean(), 100 * (K[conf] != "정상").mean(), 100 * (K[~conf] != "정상").mean()))
SUBTYPE = {k: int(v) for k, v in kc.items()}

후처리 적용 후 오답 행: 16611 (30.35%)
                          행 수  오답 대비 %
문장부호만 다름            7423        44.69
글자(어휘) 차이 있음       5038        30.33
띄어쓰기만 다름            3131        18.85
띄어쓰기+문장부호만 다름   1019         6.13

문장부호만 다른 오답의 (정답 끝부호 → 예측 끝부호) 상위 10: [(('.', '(없음)'), 1306), (('(없음)', '(없음)'), 1082), (('(없음)', '.'), 1079), (('.', '.'), 1051), ((',', '.'), 870), (('.', ','), 339), ((',', '(없음)'), 326), (('?', '.'), 248), (('?', '?'), 219), (('.', '?'), 182)]

같은 입력에 서로 다른 정답이 Validation 안에 존재하는 행: 3090 (5.65%) | 그 행의 오답률 33.46% vs 나머지 30.16%


In [9]:
# [셀 6] 그림 + 저장 (Failure-type distribution, error rate by length)
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
en = {"정상": "Correct", "형식 변환만(raw 오답)": "Format-only diff", "과교정": "Over-correction", "미교정": "Missed", "부분교정": "Partial fix", "잘못된 교정": "Wrong fix", "내용 추가·삭제": "Content add/del"}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8)); errs = TB.drop(index="정상")["행 수"]
ax[0].barh([en[k] for k in errs.index][::-1], errs.values[::-1], color="#d9822b"); ax[0].set_xlabel("Rows"); ax[0].set_title("Validation error types (pko-T5 Greedy)")
for i, v in enumerate(errs.values[::-1]): ax[0].text(v, i, f" {v:,}", va="center", fontsize=8)
g = pd.DataFrame({"len": pd.cut(lens, [-1, 5, 10, 20, 40, 10**6], labels=["<=5", "6-10", "11-20", "21-40", ">40"]), "e": REAL}).groupby("len", observed=True)["e"].mean() * 100
ax[1].bar(g.index.astype(str), g.values, color="#4a7ab5"); ax[1].set_xlabel("Input length (characters)"); ax[1].set_ylabel("Substantive error rate (%)"); ax[1].set_title("Error rate by input length")
plt.tight_layout(); plt.savefig(P.REPORTS / "fig12_error_analysis.png", dpi=150); plt.close()
pd.DataFrame({"utterance_id": [r["utterance_id"] for r in VAL], "category": CAT, "input": SRC, "target": TGT, "prediction": PRED}).query("category != '정상'").to_csv(P.RUNS / "validation_errors_12.csv", index=False, encoding="utf-8-sig")
(P.RUNS / "error_analysis_12.json").write_text(json.dumps({"postprocess_experiment": POST, "error_subtypes_after_postprocess": SUBTYPE, "counts": {k: int(v) for k, v in cnt.items()}, "substantive_error_rate": float(REAL.mean()), "diag": diag, "scope": "Validation only, rule-based automatic classification (not manually reviewed)"}, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장: reports/fig12_error_analysis.png, runs/validation_errors_12.csv, runs/error_analysis_12.json")

저장: reports/fig12_error_analysis.png, runs/validation_errors_12.csv, runs/error_analysis_12.json


## 해석
**오류 분류(전체 Validation 54,730행, 자동 규칙 기반 — 사람이 검수하지 않음)**

| 유형 | 행 수 | 전체 대비 |
|---|---|---|
| 정상 | 28,838 | 52.69% |
| 형식 변환만(raw 오답) | 9,281 | 16.96% |
| 부분교정 | 10,651 | 19.46% |
| 잘못된 교정 | 3,286 | 6.00% |
| 미교정 | 1,369 | 2.50% |
| 과교정 | 956 | 1.75% |
| 내용 추가·삭제 | 349 | 0.64% |

- **가장 큰 단일 원인은 모델 성능이 아니라 tokenizer 표현 문제다.** ‘형식 변환만’ 9,281행(전체 17.0%, 오류의 35.9%)은 의미가 같은데 문자열만 다르다. 사례: 정답 `ㅋㅋㅋㅋㅋㅋ` ↔ 예측 `ᄏᄏᄏᄏᄏᄏ`(호환 자모 U+314B → 초성 U+110F). pko-T5 SentencePiece가 입력을 NFKC로 정규화하기 때문이라는 것은 **추정 원인**이며, tokenizer 설정을 직접 열어 검증한 것은 아니다(이 환경에 transformers 없음 → 실행 후 확인). 다만 규칙 기반 복원(`ko_postprocess.restore_compat_jamo`)만으로 raw 오답이 아래처럼 사라졌으므로 인과 방향은 강하게 뒷받침된다.
- **후처리 실험(Validation, 모델 재학습 없음)**: 예측 12,893행(23.6%)이 바뀌었고, raw Exact Match 0.527→0.690, raw Balanced EM 0.565→0.766(개선 +0.201, CI95 0.193~0.209), raw 과교정률 38.3%→13.0%. 정답이었다가 틀리게 된 행은 3건, 틀렸다가 정답이 된 행은 8,908건이다. 후처리 후 raw 값이 NFKC 값(0.770)과 거의 같아졌고, 남은 ‘형식 변환만’은 376행이다(예: `ㅇㅡ`처럼 자모가 음절로 합쳐진 경우 등은 복원 불가). **부작용: 3건 악화 외에 관찰되지 않았으나, 3건이 왜 악화했는지는 확인하지 않았다.** 채택 여부는 13번에서 결정한다.
- **‘과교정’(956행, 1.75%)과 ‘미교정’(1,369행, 2.50%)은 오히려 드물다.** 지수 8~12단계에서 ‘과교정률 38%’가 크게 보였던 것은 대부분 위 형식 변환 때문이며, 실질 과교정률(NFKC, 원문 유지 행 기준)은 12.9%다. 다만 12.9%도 원문이 맞는 문장의 약 8분의 1을 바꾼다는 뜻이라 낮지 않다. 사례 `name1 → name1,`, `아기 = 천사 → 아기 = 천사.`는 문장 끝 부호를 붙이는 습관으로 보이며, 이는 학습 데이터의 부호 규칙을 반영했을 가능성이 있으나 이번에 검증하지 않았다.
- **실질 오류(형식 변환만·정상 제외) 16,611행(30.35%) 중 성격**(후처리 적용 후 분석): 문장부호만 다름 7,423(44.7%), 띄어쓰기만 다름 3,131(18.9%), 띄어쓰기+문장부호만 다름 1,019(6.1%), **글자(어휘) 차이가 있는 오류 5,038(30.3%, 전체의 9.2%)**. 즉 오류의 약 70%는 어휘 선택이 아니라 문장부호·띄어쓰기 표기에서 갈린다. 문장부호만 다른 오답에서 가장 흔한 조합은 정답 `.`↔ 예측 끝부호 없음(1,306), 정답 끝부호 없음↔예측 `.`(1,079), 정답 `,`↔예측 `.`(870)이다. 정답 라벨 자체가 부호를 일관되게 붙이지 않았을 가능성(예: 사례 `거짓말! → 거짓말`, `웅 → 웅,`)이 있으나, 같은 입력에 서로 다른 정답이 있는 행(3,090행, 5.65%)의 오답률(33.5%)이 나머지(30.2%)보다 3%p만 높아 **‘라벨 부호 불일치가 오류의 주원인’이라고 단정할 근거는 부족하다**(추가 라벨 감사 필요).
- **조건별 반복 패턴**: 입력이 길수록(≤5자 26.8% → 40자 초과 55.8%), 정답 편집 거리가 클수록(0: 12.9% → ≥5: 53.9%), 변경 유형이 `spelling_or_word`일수록(47.6%) 오류율이 높다. 부분교정은 평균 입력 길이가 18.8자로 가장 길다 → 긴 문장에서 여러 곳을 동시에 고쳐야 할 때 일부만 고치는 경향. **원인이 데이터 부족인지 max_length(72 token, Train 잘림 약 7~9%)인지 모델 용량인지는 이 분석으로 분리되지 않는다.**
- **길이·반복 진단**: 빈 출력 0, 정답 대비 150% 초과 긴 출력 128행, 50% 미만 짧은 출력 38행, 반복 패턴 9행(정답엔 없음). 모두 전체의 0.3% 이하로 희소하다.
- **라벨 문제 후보와 오류**: 숫자가 바뀐 라벨 74행의 오류율 79.7%, 영문이 바뀐 라벨 12행 91.7%, 이모지 41행 68.3%로 전체(30.4%)보다 매우 높다 → 이 행들의 오류 상당수는 모델이 아니라 **라벨 변경(예: 05번에서 확인한 `ㄷㄷㄷ→eee`류)** 때문일 가능성이 높다. 다만 표본이 작아(합계 127행, 전체의 0.23%) 전체 오류에 미치는 영향은 작다.
- **대표 사례 중 눈에 띈 정답 라벨 문제**: `아는분이 요리고수신가봐요 → 아는 분이 여리고 수신가 봐요`(정답이 어색), `두개곰 → 두 개로`. 이런 라벨은 모델이 ‘틀린’ 것으로 계산된다. 빈도는 측정하지 않았다.
- **해결하지 못한 한계**: (1) 위 분류는 규칙 기반이라 ‘부분교정’과 ‘잘못된 교정’의 경계가 편집 거리에 의존한다. (2) 환각(사실과 다른 내용 생성)을 별도 유형으로 자동 측정하지 못했다(‘내용 추가·삭제’ 349행이 가장 가까운 근사). (3) 실패 사례는 무작위 4건씩만 제시했으며 전수 검수하지 않았다. (4) 후처리는 Validation에서만 검증했다. (5) ET5·다른 모델과 오류 패턴을 비교하지 않았다.